# TinyGPT — stage 4: intent+memory fine-tune (corpus v6)

Resumes from the epoch-17 long-run model on corpus v6 (14.1M tokens = corpus v5
+ in-weights memory conversations + expanded social surface matching mind.py).
4 fresh passes at low LR (2.5e-5), fresh cosine, checkpoints every 250 steps.


In [ ]:
# 1) GPU check (HARD) + copy files from the mounted dataset
import glob, os, shutil, subprocess, torch

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "NO GPU on this session — set Accelerator to GPU T4 and re-run!"
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

hits = glob.glob("/kaggle/input/**/final.pt", recursive=True)
assert hits, f"dataset not mounted: {os.listdir('/kaggle/input')}"
IN = os.path.dirname(hits[0])
print("dataset found at:", IN)
shutil.copytree(IN, "/kaggle/working/ai_v2", dirs_exist_ok=True)
os.chdir("/kaggle/working/ai_v2")
print("workdir ready:", sorted(os.listdir(".")))

In [ ]:
# 2) Sanity check: what are we resuming from?
ck = torch.load("final.pt", map_location="cpu")
print("resume ckpt: epoch", ck["epoch"], "| step", ck["global_step"],
      "| vocab", ck["config"]["vocab_size"], "| layers", ck["config"]["n_layers"])

In [ ]:
# 3) TRAIN — stage 4: resume epoch-17 -> 4 fresh passes over corpus v6 (14.1M tok).
# Fine-tune mode auto-triggers (ckpt step 6399 > run total), so fresh optimizer +
# fresh cosine 2.5e-5 -> 0. ~5-6 min/epoch on the T4.
!python train.py --data_dir . --tokenizer tokenizer_chat.json \
    --resume final.pt --vocab_size 4099 \
    --context_length 512 --n_layers 5 \
    --batch_size 32 --grad_accum_steps 1 \
    --epochs 4 --lr 2.5e-5 --warmup_steps 30 \
    --eval_every 250 --eval_batches 50 \
    --checkpoint_dir checkpoints/s3


In [ ]:
# 4) Quality check — old skills + the NEW skills this stage teaches.
from model import ModelConfig, TinyGPT, generate, resolve_device
from tokenizer import load_tokenizer
import torch

tok = load_tokenizer('tokenizer_chat.json')
ck = torch.load('checkpoints/s3/final.pt', map_location='cpu')
cfg = ModelConfig(**ck['config'])
model = TinyGPT(cfg)
model.load_state_dict(ck['model_state'])
model.eval()
end_id = tok.token_to_id('<|end|>')

def ask(q, temp=0.0):
    ids = tok.encode(f'<|user|>{q}<|assistant|>').ids
    idx = torch.tensor([ids])
    out = generate(model, idx, max_new_tokens=80, temperature=temp, top_k=50,
                   eos_id=end_id, repetition_penalty=1.15)
    new = out[0].tolist()[len(ids):]
    if end_id in new: new = new[:new.index(end_id)]
    return tok.decode(new, skip_special_tokens=True).strip()

def ask2_ctx(q1, a1, q2, temp=0.0):
    ids = tok.encode(f'<|user|>{q1}<|assistant|>{a1}<|end|><|user|>{q2}<|assistant|>').ids
    idx = torch.tensor([ids])
    out = generate(model, idx, max_new_tokens=80, temperature=temp, top_k=50,
                   eos_id=end_id, repetition_penalty=1.15)
    new = out[0].tolist()[len(ids):]
    if end_id in new: new = new[:new.index(end_id)]
    return tok.decode(new, skip_special_tokens=True).strip()

def ask2(q1, q2, temp=0.0):
    a1 = ask(q1, temp)
    return ask2_ctx(q1, a1, q2, temp)

probes = [
    ('OLD FACT', 'what is the capital of France?'),
    ('OLD SOCIAL', 'hi'),
    ('NEW SOCIAL', 'thank you'),
    ('NEW SOCIAL', 'who are you'),
    ('NEW SOCIAL', 'are you a robot?'),
    ('NEW SOCIAL', 'what can you do'),
    ('NEW MEMORY-RAW', 'my name is Priya'),
    ('NEW MEMORY-RECALL', ('my name is Priya', 'what is my name?')),
    ('NEW MEMORY-LIKE', ('i like cricket', 'what do i like?')),
]
for label, q in probes:
    if isinstance(q, tuple):
        print(f'{label}: {q[0]!r} -> {ask(q[0])!r}; then {q[1]!r} -> {ask2(q[0], q[1])!r}')
    else:
        print(f'{label}: {q!r} -> {ask(q)!r}')


In [ ]:
# 5) Package the trained checkpoints -> download from the Output tab (right panel)
!zip -r /kaggle/working/stage3_checkpoints.zip checkpoints/s3
!ls -lh /kaggle/working/stage3_checkpoints.zip